# Notebook 03: Feature Engineering

**Goal:** Transform raw features into modeling-ready features for the recommendation system.

**Steps:**
1. Load data and setup
2. Fix data quality issues (replace 'add some')
3. Metadata features (studio encoding, genre features)
4. Temporal features (decade bins, age, seasonality)
5. Popularity and engagement features
6. Text length features
7. Multi-hot encoding for genres and themes
8. Target variable engineering
9. Feature validation and save

---

## 1. Setup and Load Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

DATA_DIR = Path('data')
PROCESSED_DIR = DATA_DIR / 'processed'

df = pd.read_parquet(PROCESSED_DIR / 'anime_clean.parquet')

print("Data loaded successfully")
print(f"Shape: {df.shape}")
print(f"\nCurrent features: {df.columns.tolist()[:15]}...")
print(f"\nWe will engineer features for:")
print("  - Studio encoding (categorical)")
print("  - Genre multi-hot vectors")
print("  - Temporal features")
print("  - Popularity normalization")
print("  - Text features")

Data loaded successfully
Shape: (19931, 45)

Current features: ['myanimelist_id', 'title', 'description', 'image', 'source_url', 'Type', 'Episodes', 'Status', 'Premiered', 'Released_Season', 'Released_Year', 'Source', 'Duration', 'Rating', 'Demographic']...

We will engineer features for:
  - Studio encoding (categorical)
  - Genre multi-hot vectors
  - Temporal features
  - Popularity normalization
  - Text features


## 2. Fix Data Quality Issues

Replace 'add some' placeholder with 'Unknown' in studios and producers.

In [3]:
# Fix studios
def clean_studios(x):
    if len(x) == 1 and x[0] == 'add some':
        return ['Unknown']
    return x

df['studios_list'] = df['studios_list'].apply(clean_studios)
df['Studios'] = df['Studios'].replace('add some', 'Unknown')

# Fix producers
def clean_producers(x):
    if len(x) == 1 and x[0] == 'add some':
        return []
    return x

df['producers_list'] = df['producers_list'].apply(clean_producers)
df['Producers'] = df['Producers'].replace('add some', None)

# Update counts
def count_studios(x):
    if len(x) == 1 and x[0] == 'Unknown':
        return 0
    return len(x)

df['studios_count'] = df['studios_list'].apply(count_studios)
df['producers_count'] = df['producers_list'].apply(len)

# Update primary studio
df['primary_studio'] = df['studios_list'].apply(lambda x: x[0] if len(x) > 0 else 'Unknown')

# Create indicator for known studio
df['has_known_studio'] = (df['primary_studio'] != 'Unknown').astype(int)

print("Data quality fixes applied")
print(f"\nStudio distribution (top 10):")
print(df['primary_studio'].value_counts().head(10))
print(f"\nAnime with known studio: {df['has_known_studio'].sum()} ({df['has_known_studio'].mean()*100:.1f}%)")
print(f"Anime with unknown studio: {(1-df['has_known_studio']).sum()} ({(1-df['has_known_studio'].mean())*100:.1f}%)")

Data quality fixes applied

Studio distribution (top 10):
primary_studio
Unknown              4773
Toei Animation        824
Sunrise               535
J.C.Staff             427
TMS Entertainment     365
Madhouse              351
Production I.G        321
Studio Deen           295
OLM                   282
Studio Pierrot        268
Name: count, dtype: int64

Anime with known studio: 15158 (76.1%)
Anime with unknown studio: 4773 (23.9%)


## 3. Studio Encoding Features

Create categorical encoding for studios with frequency-based grouping.

In [4]:
# Get studio frequency
studio_freq = df['primary_studio'].value_counts()

# Define top studios (appear in at least 50 anime)
TOP_STUDIO_THRESHOLD = 50
top_studios = studio_freq[studio_freq >= TOP_STUDIO_THRESHOLD].index.tolist()

# Remove 'Unknown' from top studios list
if 'Unknown' in top_studios:
    top_studios.remove('Unknown')

print(f"Studio encoding strategy:")
print(f"  Total unique studios: {df['primary_studio'].nunique()}")
print(f"  Studios with 50+ anime: {len(top_studios)}")
print(f"  Top studios: {top_studios[:15]}")

# Create studio category feature
def categorize_studio(studio):
    if studio == 'Unknown':
        return 'Unknown'
    elif studio in top_studios:
        return studio
    else:
        return 'Other'

df['studio_category'] = df['primary_studio'].apply(categorize_studio)

# Studio quality score (average score by studio)
studio_quality = df[df['Score'].notna()].groupby('primary_studio')['Score'].mean()
df['studio_avg_score'] = df['primary_studio'].map(studio_quality).fillna(df['Score'].mean())

# Studio popularity (average members by studio)
studio_popularity = df.groupby('primary_studio')['log_members'].mean()
df['studio_avg_popularity'] = df['primary_studio'].map(studio_popularity).fillna(df['log_members'].mean())

print(f"\nStudio categories created:")
print(df['studio_category'].value_counts().head(10))

print(f"\nNew studio features:")
print(f"  studio_category: categorical feature")
print(f"  studio_avg_score: numeric (mean: {df['studio_avg_score'].mean():.3f})")
print(f"  studio_avg_popularity: numeric (mean: {df['studio_avg_popularity'].mean():.3f})")

Studio encoding strategy:
  Total unique studios: 1057
  Studios with 50+ anime: 67
  Top studios: ['Toei Animation', 'Sunrise', 'J.C.Staff', 'TMS Entertainment', 'Madhouse', 'Production I.G', 'Studio Deen', 'OLM', 'Studio Pierrot', 'Shin-Ei Animation', 'A-1 Pictures', 'Nippon Animation', 'DLE', 'AIC', 'Tatsunoko Production']

Studio categories created:
studio_category
Other                6085
Unknown              4773
Toei Animation        824
Sunrise               535
J.C.Staff             427
TMS Entertainment     365
Madhouse              351
Production I.G        321
Studio Deen           295
OLM                   282
Name: count, dtype: int64

New studio features:
  studio_category: categorical feature
  studio_avg_score: numeric (mean: 6.426)
  studio_avg_popularity: numeric (mean: 8.230)


## Feature Strategy Discussion

**Studio alone is NOT enough for recommendations. Here's our multi-modal approach:**

### 1. **Metadata Features** (what we're building now)
- Studio: Quality signal, production style indicator
- Genre: Content similarity (most important!)
- Type: Format preference (TV vs Movie vs OVA)
- Demographic: Target audience alignment
- Themes: Specific interests (mecha, school, isekai)

### 2. **Text Embeddings** (Notebook 04)
- Description embeddings using sentence-transformers
- Captures plot, tone, themes semantically
- Most powerful for content-based similarity

### 3. **Image Embeddings** (Notebook 05)
- Cover art style using CLIP
- Visual aesthetic preferences
- Art style similarity

### 4. **Graph Features** (Notebook 06)
- Anime-Studio-Producer relationships
- Shared staff connections
- Franchise/sequel networks

### 5. **Engagement Features**
- Popularity (Members, Favorites)
- Quality (Score, Ranked)
- Community preference signals

### Final Recommendation System:
**Hybrid Model = Text Embeddings (40%) + Metadata (30%) + Graph (15%) + Images (10%) + Engagement (5%)**

Studio is just ONE signal among many. Let's continue building all features!

In [5]:
print("FEATURE IMPORTANCE HIERARCHY (Planned)")
print("="*60)
print("\n1. TEXT EMBEDDINGS (40% weight)")
print("   - Description semantic similarity")
print("   - Most accurate content matching")

print("\n2. METADATA FEATURES (30% weight)")
print("   - Genre multi-hot vectors (primary)")
print("   - Studio quality signal")
print("   - Type, Rating, Demographic")
print("   - Themes and tags")

print("\n3. GRAPH EMBEDDINGS (15% weight)")
print("   - Studio-Producer network")
print("   - Franchise connections")
print("   - Staff overlap")

print("\n4. IMAGE EMBEDDINGS (10% weight)")
print("   - Visual style similarity")
print("   - Art aesthetic matching")

print("\n5. ENGAGEMENT SIGNALS (5% weight)")
print("   - Popularity normalization")
print("   - Score adjustment")
print("   - Community favorites")

print("\n" + "="*60)
print("Current step: Building metadata features foundation")
print("Next: Multi-hot genre encoding (most important metadata)")

FEATURE IMPORTANCE HIERARCHY (Planned)

1. TEXT EMBEDDINGS (40% weight)
   - Description semantic similarity
   - Most accurate content matching

2. METADATA FEATURES (30% weight)
   - Genre multi-hot vectors (primary)
   - Studio quality signal
   - Type, Rating, Demographic
   - Themes and tags

3. GRAPH EMBEDDINGS (15% weight)
   - Studio-Producer network
   - Franchise connections
   - Staff overlap

4. IMAGE EMBEDDINGS (10% weight)
   - Visual style similarity
   - Art aesthetic matching

5. ENGAGEMENT SIGNALS (5% weight)
   - Popularity normalization
   - Score adjustment
   - Community favorites

Current step: Building metadata features foundation
Next: Multi-hot genre encoding (most important metadata)


## 4. Multi-Hot Genre Encoding

Create binary vectors for genres and themes for content-based similarity.

In [6]:
# Multi-hot encode genres
mlb_genres = MultiLabelBinarizer()
genre_matrix = mlb_genres.fit_transform(df['genres_list'])
genre_columns = [f'genre_{genre}' for genre in mlb_genres.classes_]
genre_df = pd.DataFrame(genre_matrix, columns=genre_columns, index=df.index)

print("Genre Multi-Hot Encoding")
print(f"  Total genres: {len(mlb_genres.classes_)}")
print(f"  Genre columns created: {len(genre_columns)}")
print(f"  Genres: {list(mlb_genres.classes_)}")

# Multi-hot encode themes
mlb_themes = MultiLabelBinarizer()
theme_matrix = mlb_themes.fit_transform(df['themes_list'])
theme_columns = [f'theme_{theme}' for theme in mlb_themes.classes_]
theme_df = pd.DataFrame(theme_matrix, columns=theme_columns, index=df.index)

print(f"\nTheme Multi-Hot Encoding")
print(f"  Total themes: {len(mlb_themes.classes_)}")
print(f"  Theme columns created: {len(theme_columns)}")
print(f"  Top themes: {list(mlb_themes.classes_)[:15]}")

# Add to main dataframe
df = pd.concat([df, genre_df, theme_df], axis=1)

print(f"\nDataframe shape after encoding: {df.shape}")
print(f"\nSample genre encoding (first anime):")
genre_cols_sample = [col for col in genre_df.columns if df[col].iloc[0] == 1]
print(f"  {df['title'].iloc[0]}: {genre_cols_sample}")

Genre Multi-Hot Encoding
  Total genres: 21
  Genre columns created: 21
  Genres: ['Action', 'Adventure', 'Avant Garde', 'Award Winning', 'Boys Love', 'Comedy', 'Drama', 'Ecchi', 'Erotica', 'Fantasy', 'Girls Love', 'Gourmet', 'Hentai', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Slice of Life', 'Sports', 'Supernatural', 'Suspense']

Theme Multi-Hot Encoding
  Total themes: 52
  Theme columns created: 52
  Top themes: ['Adult Cast', 'Anthropomorphic', 'CGDCT', 'Childcare', 'Combat Sports', 'Crossdressing', 'Delinquents', 'Detective', 'Educational', 'Gag Humor', 'Gore', 'Harem', 'High Stakes Game', 'Historical', 'Idols (Female)']

Dataframe shape after encoding: (19931, 122)

Sample genre encoding (first anime):
  Cowboy Bebop: ['genre_Action', 'genre_Award Winning', 'genre_Sci-Fi']


## 5. Temporal Features

Create time-based features including age, decade bins, and seasonality.

In [8]:
# Current year for age calculation
CURRENT_YEAR = 2025

# Anime age
df['anime_age'] = CURRENT_YEAR - df['Released_Year'].fillna(CURRENT_YEAR)
df['anime_age'] = df['anime_age'].clip(lower=0)

# Decade bins
df['decade_bin'] = pd.cut(
    df['Released_Year'].fillna(2020),
    bins=[1960, 1970, 1980, 1990, 2000, 2010, 2020, 2030],
    labels=['1960s', '1970s', '1980s', '1990s', '2000s', '2010s', '2020s']
)

# Season one-hot encoding
season_dummies = pd.get_dummies(df['Released_Season'], prefix='season')
df = pd.concat([df, season_dummies], axis=1)

# Era indicators (handle NaN properly)
df['is_classic'] = df['Released_Year'].apply(lambda x: 1 if pd.notna(x) and x < 2000 else 0)
df['is_golden_age'] = df['Released_Year'].apply(lambda x: 1 if pd.notna(x) and 2000 <= x < 2010 else 0)
df['is_modern'] = df['Released_Year'].apply(lambda x: 1 if pd.notna(x) and x >= 2010 else 0)

# Has release date info
df['has_release_info'] = df['Released_Year'].notna().astype(int)

print("Temporal Features Created")
print("="*60)

print(f"\nAnime Age:")
print(f"  Mean age: {df['anime_age'].mean():.1f} years")
print(f"  Median age: {df['anime_age'].median():.1f} years")
print(f"  Range: {df['anime_age'].min():.0f} - {df['anime_age'].max():.0f} years")

print(f"\nDecade Distribution:")
print(df['decade_bin'].value_counts().sort_index())

print(f"\nEra Distribution:")
print(f"  Classic (<2000): {df['is_classic'].sum()} ({df['is_classic'].mean()*100:.1f}%)")
print(f"  Golden Age (2000-2009): {df['is_golden_age'].sum()} ({df['is_golden_age'].mean()*100:.1f}%)")
print(f"  Modern (2010+): {df['is_modern'].sum()} ({df['is_modern'].mean()*100:.1f}%)")

print(f"\nSeason columns created: {[col for col in df.columns if col.startswith('season_')]}")
print(f"\nTotal temporal features added: 11+")

Temporal Features Created

Anime Age:
  Mean age: 4.6 years
  Median age: 0.0 years
  Range: 0 - 64 years

Decade Distribution:
decade_bin
1960s       95
1970s      218
1980s      308
1990s      490
2000s     1341
2010s    16082
2020s     1397
Name: count, dtype: int64

Era Distribution:
  Classic (<2000): 1051 (5.3%)
  Golden Age (2000-2009): 1263 (6.3%)
  Modern (2010+): 3868 (19.4%)

Season columns created: ['season_Fall', 'season_Spring', 'season_Summer', 'season_Winter', 'season_Fall', 'season_Spring', 'season_Summer', 'season_Winter']

Total temporal features added: 11+


## 6. Popularity and Engagement Features

Normalize popularity metrics and create engagement signals.

In [9]:
# Popularity percentile ranks
df['members_percentile'] = df['Members'].rank(pct=True)
df['favorites_percentile'] = df['Favorites'].rank(pct=True)
df['score_percentile'] = df['Score'].rank(pct=True)

# Popularity bins
df['popularity_tier'] = pd.cut(
    df['members_percentile'],
    bins=[0, 0.25, 0.50, 0.75, 0.90, 1.0],
    labels=['Niche', 'Low', 'Medium', 'High', 'Very High']
)

# Engagement rate (favorites per member)
df['engagement_rate'] = df['favorite_rate']

# Normalized score (z-score)
df['score_normalized'] = (df['Score'] - df['Score'].mean()) / df['Score'].std()

# Score quality indicator
df['is_highly_rated'] = (df['Score'] >= 7.5).astype(int)
df['is_low_rated'] = (df['Score'] <= 6.0).astype(int)

# Popularity vs quality gap
df['popularity_quality_gap'] = df['members_percentile'] - df['score_percentile'].fillna(0.5)

# Has sufficient data for rating
df['has_sufficient_ratings'] = (df['Members'] >= 1000).astype(int)

print("Popularity & Engagement Features Created")
print("="*60)

print(f"\nPercentile Features:")
print(f"  members_percentile: mean={df['members_percentile'].mean():.3f}")
print(f"  favorites_percentile: mean={df['favorites_percentile'].mean():.3f}")
print(f"  score_percentile: mean={df['score_percentile'].mean():.3f}")

print(f"\nPopularity Tiers:")
print(df['popularity_tier'].value_counts().sort_index())

print(f"\nQuality Indicators:")
print(f"  Highly rated (>=7.5): {df['is_highly_rated'].sum()} ({df['is_highly_rated'].mean()*100:.1f}%)")
print(f"  Low rated (<=6.0): {df['is_low_rated'].sum()} ({df['is_low_rated'].mean()*100:.1f}%)")

print(f"\nSufficient ratings (>=1000 members): {df['has_sufficient_ratings'].sum()} ({df['has_sufficient_ratings'].mean()*100:.1f}%)")

print(f"\nTotal engagement features added: 10")

Popularity & Engagement Features Created

Percentile Features:
  members_percentile: mean=0.500
  favorites_percentile: mean=0.500
  score_percentile: mean=0.500

Popularity Tiers:
popularity_tier
Niche        4980
Low          4985
Medium       4983
High         2989
Very High    1994
Name: count, dtype: int64

Quality Indicators:
  Highly rated (>=7.5): 2079 (10.4%)
  Low rated (<=6.0): 4164 (20.9%)

Sufficient ratings (>=1000 members): 13524 (67.9%)

Total engagement features added: 10


## 7. Text Features

Extract text length and content indicators from description and title.

In [10]:
# Title features
df['title_length'] = df['title'].fillna('').str.len()
df['title_word_count'] = df['title'].fillna('').str.split().str.len()

# Description features
df['description_length'] = df['description'].fillna('').str.len()
df['description_word_count'] = df['description'].fillna('').str.split().str.len()
df['has_description'] = (df['description_length'] > 0).astype(int)

# Character name features (already have character_count)
df['has_characters'] = (df['character_count'] > 0).astype(int)

# Image availability
df['has_image'] = df['image'].notna().astype(int)

print("Text Features Created")
print("="*60)

print(f"\nTitle Features:")
print(f"  Average title length: {df['title_length'].mean():.1f} characters")
print(f"  Average title words: {df['title_word_count'].mean():.1f} words")

print(f"\nDescription Features:")
print(f"  Average description length: {df['description_length'].mean():.0f} characters")
print(f"  Average description words: {df['description_word_count'].mean():.0f} words")
print(f"  Has description: {df['has_description'].sum()} ({df['has_description'].mean()*100:.1f}%)")

print(f"\nContent Availability:")
print(f"  Has characters: {df['has_characters'].sum()} ({df['has_characters'].mean()*100:.1f}%)")
print(f"  Has image: {df['has_image'].sum()} ({df['has_image'].mean()*100:.1f}%)")

print(f"\nTotal text features added: 7")

Text Features Created

Title Features:
  Average title length: 27.1 characters
  Average title words: 4.4 words

Description Features:
  Average description length: 416 characters
  Average description words: 69 words
  Has description: 19875 (99.7%)

Content Availability:
  Has characters: 13519 (67.8%)
  Has image: 19931 (100.0%)

Total text features added: 7


## 8. Additional Categorical Features

One-hot encode remaining categorical variables.

In [11]:
# Type encoding
type_dummies = pd.get_dummies(df['Type'], prefix='type')
df = pd.concat([df, type_dummies], axis=1)

# Rating encoding
rating_dummies = pd.get_dummies(df['Rating'], prefix='rating')
df = pd.concat([df, rating_dummies], axis=1)

# Source encoding
source_dummies = pd.get_dummies(df['Source'], prefix='source')
df = pd.concat([df, source_dummies], axis=1)

# Status encoding
status_dummies = pd.get_dummies(df['Status'], prefix='status')
df = pd.concat([df, status_dummies], axis=1)

# Demographic encoding
demographic_dummies = pd.get_dummies(df['Demographic'], prefix='demographic')
df = pd.concat([df, demographic_dummies], axis=1)

print("Categorical Features Encoded")
print("="*60)

print(f"\nType features: {len([c for c in df.columns if c.startswith('type_')])} categories")
print(f"  Categories: {[c.replace('type_', '') for c in df.columns if c.startswith('type_')]}")

print(f"\nRating features: {len([c for c in df.columns if c.startswith('rating_')])} categories")

print(f"\nSource features: {len([c for c in df.columns if c.startswith('source_')])} categories")

print(f"\nStatus features: {len([c for c in df.columns if c.startswith('status_')])} categories")

print(f"\nDemographic features: {len([c for c in df.columns if c.startswith('demographic_')])} categories")

print(f"\nCurrent dataframe shape: {df.shape}")

Categorical Features Encoded

Type features: 7 categories
  Categories: ['Movie', 'ONA', 'OVA', 'Special', 'TV', 'TV Special', 'Unknown']

Rating features: 7 categories

Source features: 18 categories

Status features: 4 categories

Demographic features: 6 categories

Current dataframe shape: (19931, 193)


## 9. Feature Summary and Save

Review all engineered features and save the feature store.

In [13]:
# Remove duplicate columns
df = df.loc[:, ~df.columns.duplicated()]

# Count features by category
feature_categories = {
    'Original': 45,
    'Genre (multi-hot)': len([c for c in df.columns if c.startswith('genre_')]),
    'Theme (multi-hot)': len([c for c in df.columns if c.startswith('theme_')]),
    'Studio': len([c for c in df.columns if 'studio' in c.lower() and not c.startswith('Studios')]),
    'Temporal': len([c for c in df.columns if c in ['anime_age', 'is_classic', 'is_golden_age', 'is_modern', 'has_release_info']]) + len([c for c in df.columns if c.startswith('season_')]),
    'Popularity': len([c for c in df.columns if 'percentile' in c or 'popularity' in c.lower() or 'engagement' in c]),
    'Text': len([c for c in df.columns if 'title_' in c or 'description_' in c or c in ['has_characters', 'has_image', 'has_description']]),
    'Type': len([c for c in df.columns if c.startswith('type_')]),
    'Rating': len([c for c in df.columns if c.startswith('rating_')]),
    'Source': len([c for c in df.columns if c.startswith('source_')]),
    'Status': len([c for c in df.columns if c.startswith('status_')]),
    'Demographic': len([c for c in df.columns if c.startswith('demographic_')])
}

print("FEATURE ENGINEERING SUMMARY")
print("="*70)
print(f"\nTotal features: {df.shape[1]}")
print(f"Total anime: {df.shape[0]}")

print("\nFeatures by Category:")
for category, count in feature_categories.items():
    print(f"  {category:25s}: {count:3d} features")

print("\n" + "="*70)
print("\nFeature Types:")
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"  Numeric features: {len(numeric_features)}")
print(f"  Categorical features: {len(categorical_features)}")

# Save feature-engineered dataset
output_path = PROCESSED_DIR / 'anime_features.parquet'
df.to_parquet(output_path, index=False)

print("\n" + "="*70)
print(f"Feature store saved: {output_path}")
print(f"File size: {output_path.stat().st_size / (1024**2):.2f} MB")

print("\n" + "="*70)
print("NOTEBOOK 03 COMPLETE")
print("="*70)
print("\nDeliverables:")
print("  - anime_features.parquet (193 features)")
print("  - Multi-hot genre vectors (21 genres)")
print("  - Multi-hot theme vectors (52 themes)")
print("  - Studio quality signals")
print("  - Temporal features")
print("  - Engagement metrics")
print("\nNext: Notebook 04 - Text Embeddings (sentence-transformers)")

FEATURE ENGINEERING SUMMARY

Total features: 189
Total anime: 19931

Features by Category:
  Original                 :  45 features
  Genre (multi-hot)        :  21 features
  Theme (multi-hot)        :  52 features
  Studio                   :   7 features
  Temporal                 :   9 features
  Popularity               :   8 features
  Text                     :   7 features
  Type                     :   7 features
  Rating                   :   7 features
  Source                   :  18 features
  Status                   :   4 features
  Demographic              :   6 features


Feature Types:
  Numeric features: 117
  Categorical features: 27

Feature store saved: data\processed\anime_features.parquet
File size: 12.24 MB

NOTEBOOK 03 COMPLETE

Deliverables:
  - anime_features.parquet (193 features)
  - Multi-hot genre vectors (21 genres)
  - Multi-hot theme vectors (52 themes)
  - Studio quality signals
  - Temporal features
  - Engagement metrics

Next: Notebook 04 - Text 